### Rain in Australia Prediction

#### 1. Data Overview & Cleaning

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('weatherAUS.csv')

print("--- 1. DATA OVERVIEW ---")
print(f"Dimensi Dataset: {df.shape}")
print(df.info())
print("\nJumlah Missing Values:")
print(df.isnull().sum())

# Data Cleaning
# 1. Hapus baris di mana target 'RainTomorrow' bernilai NaN
df = df.dropna(subset=['RainTomorrow'])

# 2. Tangani missing values untuk kolom numerik (isi dengan median)
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# 3. Tangani missing values untuk kolom kategorikal (isi dengan modus)
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# 4. Konversi kolom tanggal menjadi fitur tambahan
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df.drop(columns=['Date'], inplace=True)

print("\nData berhasil dibersihkan. Dimensi baru:", df.shape)

--- 1. DATA OVERVIEW ---
Dimensi Dataset: (145460, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  object 
 1   Location       145460 non-null  object 
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  object 
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  object 
 10  WindDir3pm     141232 non-null  object 
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    1304

/tmp/ipykernel_429/340874843.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_429/340874843.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin

#### 2. Exploratory Data Analysis

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- 2. EXPLORATORY DATA ANALYSIS (EDA) ---")

# Distribusi Target Variable
plt.figure(figsize=(6, 4))
sns.countplot(x='RainTomorrow', data=df, palette='Set2')
plt.title('Distribusi Target: Rain Tomorrow')
plt.savefig('eda_target_distribution.png')
plt.close()

# Visualisasi Korelasi Fitur Numerik
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', fmt='.2f')
plt.title('Matriks Korelasi Fitur Numerik')
plt.savefig('eda_correlation_matrix.png')
plt.close()

print("Grafik EDA berhasil disimpan (eda_target_distribution.png & eda_correlation_matrix.png)")

--- 2. EXPLORATORY DATA ANALYSIS (EDA) ---


/tmp/ipykernel_429/2010343032.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x='RainTomorrow', data=df, palette='Set2')


Grafik EDA berhasil disimpan (eda_target_distribution.png & eda_correlation_matrix.png)


#### 3. Feature Engineering

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("--- 3. FEATURE ENGINEERING ---")

# Label Encoding untuk kolom kategorikal biner (Yes/No)
binary_cols = ['RainToday', 'RainTomorrow']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# One-Hot Encoding untuk kolom kategorikal nominal lainnya (misal: Location, WindGustDir, dll)
df = pd.get_dummies(df, columns=['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm'], drop_first=True)

# Standarisasi fitur numerik
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print(f"Dimensi setelah Feature Engineering: {df.shape}")

--- 3. FEATURE ENGINEERING ---
Dimensi setelah Feature Engineering: (142193, 114)


#### 4. Modelling & Evaluation

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("--- 4. MODELING & EVALUASI ---")

# Pisahkan fitur (X) dan target (y)
X = df.drop(columns=['RainTomorrow'])
y = df['RainTomorrow']

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Inisialisasi dan latih model
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluasi
print(f"Accuracy Score : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score  : {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

--- 4. MODELING & EVALUASI ---
Accuracy Score : 0.8577
ROC-AUC Score  : 0.8885

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.96      0.91     22064
           1       0.79      0.50      0.61      6375

    accuracy                           0.86     28439
   macro avg       0.83      0.73      0.76     28439
weighted avg       0.85      0.86      0.85     28439


Confusion Matrix:
[[21200   864]
 [ 3182  3193]]


#### 5. Overfitting & Feature Selection

In [5]:
from sklearn.ensemble import RandomForestClassifier

print("--- 5. OVERFITTING & FEATURE SELECTION ---")

# Melatih model dengan regularisasi (membatasi kedalaman pohon untuk mencegah overfitting)
tuned_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
tuned_model.fit(X_train, y_train)

print(f"Train Accuracy (Tuned): {tuned_model.score(X_train, y_train):.4f}")
print(f"Test Accuracy (Tuned) : {tuned_model.score(X_test, y_test):.4f}")

# Feature Selection Berdasarkan Feature Importance
importances = tuned_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print("\nTop 10 Fitur Paling Berpengaruh:")
print(feature_importance_df.head(10))

--- 5. OVERFITTING & FEATURE SELECTION ---
Train Accuracy (Tuned): 0.8760
Test Accuracy (Tuned) : 0.8514

Top 10 Fitur Paling Berpengaruh:
          Feature  Importance
9     Humidity3pm    0.248545
2        Rainfall    0.081317
4        Sunshine    0.074161
11    Pressure3pm    0.054924
5   WindGustSpeed    0.053847
8     Humidity9am    0.053615
10    Pressure9am    0.051850
16      RainToday    0.051654
13       Cloud3pm    0.049225
15        Temp3pm    0.034624


#### 6. Save Best Model dengan 5 Fitur

Pada tahap ini, kita membuat model khusus menggunakan lima fitur terpilih. Preprocessing dan model disimpan dalam satu file agar proses prediksi di Streamlit konsisten dengan proses training.

**Lima fitur:** `Humidity3pm`, `Cloud3pm`, `Rainfall`, `Humidity9am`, dan `RainToday`.


In [6]:
# Import library
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Baca ulang data mentah agar preprocessing dilakukan di dalam Pipeline
raw_df = pd.read_csv("weatherAUS.csv")

# Pilih 5 fitur dan target
selected_features = [
    "Humidity3pm",
    "Cloud3pm",
    "Rainfall",
    "Humidity9am",
    "RainToday"
]
target = "RainTomorrow"

model_df = raw_df[selected_features + [target]].copy()

# Hapus baris yang target-nya kosong
model_df = model_df.dropna(subset=[target])

X = model_df[selected_features]
y = model_df[target].map({"No": 0, "Yes": 1})

# Pisahkan data training dan testing
X_train_5, X_test_5, y_train_5, y_test_5 = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Pisahkan fitur numerik dan kategorikal
numeric_features = [
    "Humidity3pm",
    "Cloud3pm",
    "Rainfall",
    "Humidity9am"
]
categorical_features = ["RainToday"]

# Preprocessing numerik: isi missing value dengan median lalu standardisasi
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing kategori: isi missing value dengan modus lalu One-Hot Encoding
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Gabungkan preprocessing numerik dan kategorikal
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Buat Pipeline preprocessing + Random Forest
best_model_5_features = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Training model
best_model_5_features.fit(X_train_5, y_train_5)

# Evaluasi model 5 fitur
y_pred_5 = best_model_5_features.predict(X_test_5)
y_prob_5 = best_model_5_features.predict_proba(X_test_5)[:, 1]

print("HASIL MODEL DENGAN 5 FITUR")
print("Train Accuracy :", round(best_model_5_features.score(X_train_5, y_train_5), 4))
print("Test Accuracy  :", round(accuracy_score(y_test_5, y_pred_5), 4))
print("ROC AUC        :", round(roc_auc_score(y_test_5, y_prob_5), 4))
print("\nClassification Report:")
print(classification_report(y_test_5, y_pred_5))

# Simpan model beserta preprocessing dalam satu file PKL
joblib.dump(best_model_5_features, "weather_best_model_5_features.pkl")

print("\nModel berhasil disimpan sebagai weather_best_model_5_features.pkl")


HASIL MODEL DENGAN 5 FITUR
Train Accuracy : 0.8543
Test Accuracy  : 0.8337
ROC AUC        : 0.8252

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.95      0.90     22064
           1       0.72      0.42      0.53      6375

    accuracy                           0.83     28439
   macro avg       0.79      0.69      0.71     28439
weighted avg       0.82      0.83      0.82     28439


Model berhasil disimpan sebagai weather_best_model_5_features.pkl


#### 7. Membuat Aplikasi Streamlit

Aplikasi akan menggunakan file `weather_best_model_5_features.pkl` yang dibuat pada tahap sebelumnya. Jalankan cell berikut untuk membuat file `app.py`.


In [7]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import os

# Konfigurasi halaman
st.set_page_config(
    page_title="Prediksi Hujan Besok",
    page_icon="🌧️",
    layout="centered"
)

# Load model
MODEL_FILE = "weather_best_model_5_features.pkl"

@st.cache_resource
def load_model():
    return joblib.load(MODEL_FILE)

# Header aplikasi
st.title("🌧️ Prediksi Hujan Besok")
st.subheader("Rain in Australia Prediction")
st.write(
    "Masukkan kondisi cuaca hari ini untuk memperkirakan "
    "apakah besok akan terjadi hujan."
)

# Pastikan file model tersedia
if not os.path.exists(MODEL_FILE):
    st.error(
        "File model belum ditemukan. Jalankan cell Save Best Model "
        "di notebook terlebih dahulu."
    )
    st.stop()

model = load_model()

# Form input pengguna
st.header("Form Input Kondisi Cuaca")

with st.form("weather_form"):
    humidity_3pm = st.number_input(
        "Kelembapan pukul 15:00 (%)",
        min_value=0.0,
        max_value=100.0,
        value=50.0
    )

    cloud_3pm = st.number_input(
        "Tutupan awan pukul 15:00 (okta)",
        min_value=0.0,
        max_value=10.0,
        value=5.0
    )

    rainfall = st.number_input(
        "Curah hujan hari ini (mm)",
        min_value=0.0,
        value=0.0
    )

    humidity_9am = st.number_input(
        "Kelembapan pukul 09:00 (%)",
        min_value=0.0,
        max_value=100.0,
        value=60.0
    )

    rain_today = st.selectbox(
        "Apakah hari ini hujan?",
        options=["No", "Yes"]
    )

    submitted = st.form_submit_button("Prediksi Cuaca Besok")

# Proses prediksi
if submitted:
    input_data = pd.DataFrame([{
        "Humidity3pm": humidity_3pm,
        "Cloud3pm": cloud_3pm,
        "Rainfall": rainfall,
        "Humidity9am": humidity_9am,
        "RainToday": rain_today
    }])

    prediction = model.predict(input_data)[0]
    probabilities = model.predict_proba(input_data)[0]
    rain_probability = probabilities[list(model.classes_).index(1)]

    st.header("Hasil Prediksi")

    if prediction == 1:
        st.error("⚠️ Prediksi: BESOK HUJAN")
    else:
        st.success("☀️ Prediksi: BESOK TIDAK HUJAN")

    st.write(
        f"Probabilitas hujan: **{rain_probability * 100:.2f}%**"
    )
    st.progress(float(rain_probability))

    st.caption(
        "Hasil ini merupakan prediksi berdasarkan model machine learning, "
        "bukan kepastian kondisi cuaca."
    )


Writing app.py


#### 8. Menjalankan Aplikasi Streamlit

Pastikan file berikut berada dalam satu folder:

- `app.py`
- `weather_best_model_5_features.pkl`

Install library yang diperlukan, lalu jalankan aplikasi melalui terminal.


In [ ]:
# Jalankan perintah ini di terminal / Command Prompt, bukan di cell Python:
# pip install streamlit pandas scikit-learn joblib
# streamlit run app.py


#### 9. Kesimpulan Deployment

1. Model Random Forest dengan lima fitur telah dilatih ulang.
2. Preprocessing dan model disimpan bersama dalam satu file `.pkl`.
3. Streamlit menerima lima input yang sama dengan fitur saat training.
4. Tombol prediksi menampilkan hasil prediksi hujan atau tidak hujan beserta probabilitasnya.

**Catatan:** nilai Accuracy dan ROC AUC model lima fitur harus diambil dari hasil eksekusi notebook. Jangan menggunakan nilai evaluasi model sebelumnya karena model dan jumlah fiturnya berbeda.
